# NYC Taxi Trip Duration Prediction

## Problem Statement
Predict how long a taxi ride will take in New York City based on pickup/dropoff locations, time of day, and other trip features. This is a **regression** problem — we're predicting a continuous value (trip duration in seconds).

## Why This Matters
Ride-hailing services like Uber and Lyft need accurate trip duration estimates to:
- Assign drivers efficiently
- Provide accurate ETAs to passengers
- Optimize pricing and route planning

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully!")

In [ ]:
df = pd.read_csv('nyc_taxi_trip_duration.csv')

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
print("Data Types:")
print(df.dtypes)
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

In [ ]:
df.describe()

---
## 2. Exploratory Data Analysis (EDA)

### 2.1 Target Variable: Trip Duration

In [ ]:
print("Trip Duration Statistics:")
print(f"  Minimum:  {df['trip_duration'].min():>10,}s  ({df['trip_duration'].min()/60:.1f} min)")
print(f"  25th pct: {df['trip_duration'].quantile(0.25):>10,.0f}s  ({df['trip_duration'].quantile(0.25)/60:.1f} min)")
print(f"  Median:   {df['trip_duration'].median():>10,.0f}s  ({df['trip_duration'].median()/60:.1f} min)")
print(f"  Mean:     {df['trip_duration'].mean():>10,.0f}s  ({df['trip_duration'].mean()/60:.1f} min)")
print(f"  75th pct: {df['trip_duration'].quantile(0.75):>10,.0f}s  ({df['trip_duration'].quantile(0.75)/60:.1f} min)")
print(f"  Maximum:  {df['trip_duration'].max():>10,}s  ({df['trip_duration'].max()/3600:.1f} hours)")
print(f"\nTrips over 2 hours: {(df['trip_duration'] > 7200).sum():,} ({(df['trip_duration'] > 7200).mean()*100:.2f}%)")
print(f"Trips under 1 min:  {(df['trip_duration'] < 60).sum():,} ({(df['trip_duration'] < 60).mean()*100:.2f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution (capped for visibility)
axes[0].hist(df['trip_duration'].clip(upper=5000), bins=100, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Trip Duration (seconds)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Trip Duration Distribution (capped at 5000s)')
axes[0].axvline(df['trip_duration'].median(), color='red', linestyle='--', label=f'Median: {df["trip_duration"].median():.0f}s')
axes[0].legend()

# Log-transformed distribution
axes[1].hist(np.log1p(df['trip_duration']), bins=100, color='coral', edgecolor='white')
axes[1].set_xlabel('Log(Trip Duration)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log-Transformed Trip Duration')

plt.tight_layout()
plt.show()

print("The log-transformed distribution is much more symmetric — better for modeling!")

### 2.2 Temporal Patterns

In [ ]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'])

df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek
df['month'] = df['pickup_datetime'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Trips by hour
df.groupby('hour')['trip_duration'].median().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Median Trip Duration by Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Median Duration (s)')

# Trips by day of week
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
day_medians = df.groupby('day_of_week')['trip_duration'].median()
axes[1].bar(range(7), day_medians.values, color='coral', tick_label=day_names)
axes[1].set_title('Median Trip Duration by Day of Week')
axes[1].set_ylabel('Median Duration (s)')

# Trip count by hour
df.groupby('hour')['id'].count().plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Number of Trips by Hour')
axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Trip Count')

plt.tight_layout()
plt.show()

### 2.3 Passenger Count & Vendor

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Passenger count distribution
df['passenger_count'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Trips by Passenger Count')
axes[0].set_xlabel('Passenger Count')
axes[0].set_ylabel('Number of Trips')

# Vendor comparison
df.groupby('vendor_id')['trip_duration'].median().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Median Trip Duration by Vendor')
axes[1].set_xlabel('Vendor ID')
axes[1].set_ylabel('Median Duration (s)')

plt.tight_layout()
plt.show()

print(f"Trips with 0 passengers: {(df['passenger_count'] == 0).sum():,}")
print(f"store_and_fwd_flag: {df['store_and_fwd_flag'].value_counts().to_dict()}")

### 2.4 Geographic Distribution

In [ ]:
# Sample for plotting (full dataset is too large)
sample = df.sample(5000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(sample['pickup_longitude'], sample['pickup_latitude'],
                alpha=0.3, s=1, c='steelblue')
axes[0].set_title('Pickup Locations')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_xlim(-74.05, -73.75)
axes[0].set_ylim(40.60, 40.90)

axes[1].scatter(sample['dropoff_longitude'], sample['dropoff_latitude'],
                alpha=0.3, s=1, c='coral')
axes[1].set_title('Dropoff Locations')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_xlim(-74.05, -73.75)
axes[1].set_ylim(40.60, 40.90)

plt.tight_layout()
plt.show()

print("Most trips are concentrated in Manhattan and surrounding boroughs.")

---
## 3. Data Cleaning & Outlier Removal

In [ ]:
print(f"Original dataset size: {len(df):,} rows")

# Remove extreme trip durations
df_clean = df[(df['trip_duration'] >= 60) & (df['trip_duration'] <= 7200)].copy()
print(f"After removing trips < 1 min or > 2 hours: {len(df_clean):,} rows")

# Remove trips with 0 passengers
df_clean = df_clean[df_clean['passenger_count'] > 0]
print(f"After removing 0-passenger trips: {len(df_clean):,} rows")

# Remove geographic outliers (outside NYC bounding box)
nyc_bounds = {
    'lat_min': 40.60, 'lat_max': 40.90,
    'lon_min': -74.05, 'lon_max': -73.75
}

df_clean = df_clean[
    (df_clean['pickup_latitude'].between(nyc_bounds['lat_min'], nyc_bounds['lat_max'])) &
    (df_clean['pickup_longitude'].between(nyc_bounds['lon_min'], nyc_bounds['lon_max'])) &
    (df_clean['dropoff_latitude'].between(nyc_bounds['lat_min'], nyc_bounds['lat_max'])) &
    (df_clean['dropoff_longitude'].between(nyc_bounds['lon_min'], nyc_bounds['lon_max']))
]
print(f"After removing geographic outliers: {len(df_clean):,} rows")

# Drop columns not available at prediction time
df_clean = df_clean.drop(columns=['id', 'dropoff_datetime'])

pct_removed = (1 - len(df_clean) / len(df)) * 100
print(f"\nTotal rows removed: {len(df) - len(df_clean):,} ({pct_removed:.1f}%)")
print(f"Clean dataset: {len(df_clean):,} rows x {df_clean.shape[1]} columns")

---
## 4. Feature Engineering

We'll create new features that capture important information for predicting trip duration.

In [ ]:
# --- DateTime Features ---
df_clean['is_weekend'] = (df_clean['day_of_week'] >= 5).astype(int)

def get_time_period(hour):
    if 6 <= hour < 10:
        return 'morning_rush'
    elif 10 <= hour < 16:
        return 'midday'
    elif 16 <= hour < 20:
        return 'evening_rush'
    else:
        return 'night'

df_clean['time_period'] = df_clean['hour'].apply(get_time_period)

print("Time period distribution:")
print(df_clean['time_period'].value_counts())

In [ ]:
# --- Distance Feature (Haversine formula) ---
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two GPS coordinates."""
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df_clean['distance_km'] = haversine_distance(
    df_clean['pickup_latitude'], df_clean['pickup_longitude'],
    df_clean['dropoff_latitude'], df_clean['dropoff_longitude']
)

print(f"Distance statistics (km):")
print(df_clean['distance_km'].describe())

# --- Bearing/Direction Feature ---
def bearing(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return np.degrees(np.arctan2(x, y))

df_clean['bearing'] = bearing(
    df_clean['pickup_latitude'], df_clean['pickup_longitude'],
    df_clean['dropoff_latitude'], df_clean['dropoff_longitude']
)

print(f"\nNew features created: distance_km, bearing")

In [ ]:
# Distance vs Duration scatter plot
sample_clean = df_clean.sample(5000, random_state=42)

plt.figure(figsize=(10, 6))
plt.scatter(sample_clean['distance_km'], sample_clean['trip_duration']/60,
            alpha=0.3, s=5, c='steelblue')
plt.xlabel('Distance (km)')
plt.ylabel('Trip Duration (minutes)')
plt.title('Trip Duration vs. Distance')
plt.show()

corr = df_clean['distance_km'].corr(df_clean['trip_duration'])
print(f"Correlation between distance and trip duration: {corr:.3f}")
print("Distance is a strong predictor — longer distance = longer trip!")

In [ ]:
# --- Encode Categorical Features ---
df_clean['store_and_fwd_flag'] = (df_clean['store_and_fwd_flag'] == 'Y').astype(int)

# One-hot encode time_period
df_clean = pd.get_dummies(df_clean, columns=['time_period'], drop_first=True)

# Drop original datetime column (we extracted the features we need)
df_clean = df_clean.drop(columns=['pickup_datetime'])

print(f"Final feature set: {df_clean.shape[1] - 1} features + 1 target")
print(f"\nColumns: {list(df_clean.columns)}")

In [ ]:
# --- Log-Transform Target ---
# Trip duration is right-skewed, log transform helps models perform better
df_clean['log_trip_duration'] = np.log1p(df_clean['trip_duration'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_clean['trip_duration']/60, bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Original Trip Duration')
axes[0].set_xlabel('Minutes')

axes[1].hist(df_clean['log_trip_duration'], bins=80, color='coral', edgecolor='white')
axes[1].set_title('Log-Transformed Trip Duration')
axes[1].set_xlabel('Log(seconds)')

plt.tight_layout()
plt.show()
print("Log transformation makes the target more normally distributed.")

---
## 5. Train-Test Split

In [ ]:
# Features and target
feature_cols = [c for c in df_clean.columns if c not in ['trip_duration', 'log_trip_duration']]
X = df_clean[feature_cols]
y = df_clean['log_trip_duration']  # Predict log-transformed duration

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set:     {X_test.shape[0]:,} samples")
print(f"Features:     {X_train.shape[1]}")
print(f"\nFeature list: {feature_cols}")

---
## 6. Model Building

We'll train three regression models:
1. **Linear Regression** — simple baseline
2. **Decision Tree Regressor** — captures non-linear relationships
3. **Random Forest Regressor** — ensemble of decision trees (expected best)

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    """Train, predict, and evaluate a regression model."""
    model.fit(X_train, y_train)
    
    # Predictions (in log space)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Convert back to original scale (seconds)
    y_test_actual = np.expm1(y_test)
    y_test_pred_actual = np.expm1(y_test_pred)
    
    # Metrics in original scale
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_test_pred_actual))
    mae = mean_absolute_error(y_test_actual, y_test_pred_actual)
    r2_log = r2_score(y_test, y_test_pred)
    r2_orig = r2_score(y_test_actual, y_test_pred_actual)
    
    # Train R² for overfitting check
    r2_train = r2_score(y_train, y_train_pred)
    
    print(f"\n{'='*50}")
    print(f" {name}")
    print(f"{'='*50}")
    print(f"  Train R² (log scale):  {r2_train:.4f}")
    print(f"  Test R²  (log scale):  {r2_log:.4f}")
    print(f"  Test R²  (original):   {r2_orig:.4f}")
    print(f"  RMSE (seconds):        {rmse:.1f}  ({rmse/60:.1f} min)")
    print(f"  MAE  (seconds):        {mae:.1f}  ({mae/60:.1f} min)")
    
    if r2_train - r2_log > 0.05:
        print(f"  ⚠ Possible overfitting (train-test gap: {r2_train - r2_log:.4f})")
    
    return {
        'name': name, 'model': model,
        'r2_log': r2_log, 'r2_orig': r2_orig, 'r2_train': r2_train,
        'rmse': rmse, 'mae': mae,
        'y_test_pred': y_test_pred_actual
    }

In [ ]:
# Model 1: Linear Regression
lr_results = evaluate_model(
    "Linear Regression",
    LinearRegression(),
    X_train, X_test, y_train, y_test
)

In [ ]:
# Model 2: Decision Tree
dt_results = evaluate_model(
    "Decision Tree",
    DecisionTreeRegressor(max_depth=15, min_samples_leaf=20, random_state=42),
    X_train, X_test, y_train, y_test
)

In [ ]:
# Model 3: Random Forest
rf_results = evaluate_model(
    "Random Forest",
    RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_leaf=10,
                          n_jobs=-1, random_state=42),
    X_train, X_test, y_train, y_test
)

---
## 7. Model Evaluation & Comparison

In [ ]:
# Comparison table
results = [lr_results, dt_results, rf_results]

comparison = pd.DataFrame({
    'Model': [r['name'] for r in results],
    'Train R²': [r['r2_train'] for r in results],
    'Test R² (log)': [r['r2_log'] for r in results],
    'Test R² (orig)': [r['r2_orig'] for r in results],
    'RMSE (sec)': [r['rmse'] for r in results],
    'MAE (sec)': [r['mae'] for r in results],
    'RMSE (min)': [r['rmse']/60 for r in results],
    'MAE (min)': [r['mae']/60 for r in results],
}).set_index('Model')

print("Model Comparison:")
comparison.style.format('{:.4f}').highlight_max(subset=['Test R² (log)', 'Test R² (orig)'], color='lightgreen').highlight_min(subset=['RMSE (sec)', 'MAE (sec)'], color='lightgreen')

In [ ]:
# R² Score comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = [r['name'] for r in results]
colors = ['steelblue', 'coral', 'seagreen']

# R² scores
r2_scores = [r['r2_log'] for r in results]
bars = axes[0].bar(models, r2_scores, color=colors)
axes[0].set_title('Test R² Score (higher is better)')
axes[0].set_ylabel('R² Score')
axes[0].set_ylim(0, 1)
for bar, score in zip(bars, r2_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{score:.4f}', ha='center', fontweight='bold')

# RMSE comparison
rmse_vals = [r['rmse']/60 for r in results]
bars = axes[1].bar(models, rmse_vals, color=colors)
axes[1].set_title('RMSE in Minutes (lower is better)')
axes[1].set_ylabel('RMSE (minutes)')
for bar, val in zip(bars, rmse_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 7.1 Actual vs. Predicted

In [ ]:
y_test_actual = np.expm1(y_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, r, color in zip(axes, results, colors):
    # Sample for plotting
    idx = np.random.RandomState(42).choice(len(y_test_actual), 3000, replace=False)
    actual_sample = y_test_actual.values[idx] / 60
    pred_sample = r['y_test_pred'][idx] / 60
    
    ax.scatter(actual_sample, pred_sample, alpha=0.2, s=5, c=color)
    max_val = max(actual_sample.max(), pred_sample.max())
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1, label='Perfect prediction')
    ax.set_xlabel('Actual Duration (min)')
    ax.set_ylabel('Predicted Duration (min)')
    ax.set_title(f'{r["name"]}\nR²={r["r2_log"]:.4f}')
    ax.set_xlim(0, 60)
    ax.set_ylim(0, 60)
    ax.legend()

plt.tight_layout()
plt.show()
print("Points closer to the red dashed line = better predictions.")

### 7.2 Residual Analysis

In [ ]:
# Residuals for the best model
best = max(results, key=lambda r: r['r2_log'])
residuals = (y_test_actual.values - best['y_test_pred']) / 60  # in minutes

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual distribution
axes[0].hist(residuals.clip(-30, 30), bins=80, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual (minutes)')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'Residual Distribution — {best["name"]}')

# Residuals vs predicted
idx = np.random.RandomState(42).choice(len(residuals), 3000, replace=False)
axes[1].scatter(best['y_test_pred'][idx]/60, residuals[idx],
                alpha=0.2, s=5, c='steelblue')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Duration (min)')
axes[1].set_ylabel('Residual (min)')
axes[1].set_title('Residuals vs. Predicted')
axes[1].set_ylim(-30, 30)

plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.2f} min (should be near 0)")
print(f"Std of residuals: {residuals.std():.2f} min")

### 7.3 Feature Importance

In [ ]:
# Feature importance from Random Forest
rf_model = rf_results['model']
importance = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
importance.plot(kind='barh', color='steelblue')
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
for feat, imp in importance.tail(5).items():
    print(f"  {feat:<30s} {imp:.4f} ({imp*100:.1f}%)")

---
## 8. Best Model Selection

In [ ]:
best = max(results, key=lambda r: r['r2_log'])

print("=" * 60)
print(f"  BEST MODEL: {best['name']}")
print("=" * 60)
print(f"\n  R² Score (log scale):    {best['r2_log']:.4f}")
print(f"  R² Score (original):     {best['r2_orig']:.4f}")
print(f"  RMSE:                    {best['rmse']:.1f} seconds ({best['rmse']/60:.1f} minutes)")
print(f"  MAE:                     {best['mae']:.1f} seconds ({best['mae']/60:.1f} minutes)")
print(f"\n  On average, our model's predictions are off by ~{best['mae']/60:.0f} minutes.")

In [ ]:
# Sample predictions
sample_idx = np.random.RandomState(42).choice(len(y_test_actual), 10, replace=False)

print("Sample Predictions vs Actual:")
print(f"{'':>4} {'Actual':>12} {'Predicted':>12} {'Error':>12}")
print("-" * 44)

for i, idx in enumerate(sample_idx):
    actual = y_test_actual.values[idx] / 60
    predicted = best['y_test_pred'][idx] / 60
    error = predicted - actual
    print(f"{i+1:>3}. {actual:>9.1f} min {predicted:>9.1f} min {error:>+9.1f} min")

---
## 9. Summary & Conclusions

### What We Did
1. **Explored** 729K NYC taxi trip records with pickup/dropoff coordinates and timestamps
2. **Cleaned** the data by removing outliers (extreme durations, invalid coordinates, 0-passenger trips)
3. **Engineered features** including:
   - Haversine distance between pickup and dropoff
   - Bearing/direction of travel
   - Time-based features (hour, day of week, rush hour indicators)
4. **Built three regression models** and compared their performance

### Key Findings
- **Distance** is the single most important predictor of trip duration
- **Time of day** and **day of week** also matter (rush hour trips take longer)
- **Random Forest** outperforms both Linear Regression and Decision Trees
- Log-transforming the target variable significantly improved model performance

### Business Implications
- Dispatchers can use this model to estimate when a driver will be free
- Rush hour trips should have longer ETA estimates
- Distance-based features are essential for any trip duration prediction system